# Agente AlphaZero para Gomoku e Pente com CNN e ResNet

## 1. Introdução

Neste trabalho desenvolvemos um agente de jogo automático baseado no algoritmo **AlphaZero** para os jogos **Gomoku** e **Pente**, utilizando tabuleiro de tamanho 15×15.

O objetivo principal foi implementar uma solução completa que inclui:
- representação interna dos jogos e das respetivas regras  
- redes neuronais (uma CNN e uma variante ResNet) para política e valor  
- algoritmo de **Monte Carlo Tree Search (MCTS)** guiado pelas redes  
- ciclo de **self-play** para treinar e comparar diferentes agentes

Além disso, organizámos o código de forma modular, permitindo:
- jogar contra humanos  
- jogar contra um jogador aleatório e contra versões MCTS do agente  
- guardar modelos treinados e resultados de testes para análise posterior

---

## 2. Principais ficheiros e funções em Python

Nesta secção resumimos a estrutura do código e as funções/métodos principais usados no projeto.

### `board.py` — lógica de Gomoku & Pente

- **`class Board`**
  - `__init__(size=15, game="gomoku")`: cria o tabuleiro, define jogo (`gomoku`/`pente`), inicializa capturas e jogador atual.
  - `legal_moves()`: gera todas as casas vazias (jogadas legais).
  - `is_legal(move)`: verifica se uma jogada é válida.
  - `encode_for_player(player_id)`: recodifica o tabuleiro para que `1` seja sempre o jogador indicado.
  - `get_board_state()`: devolve o tabuleiro visto pelo jogador atual (sempre `1`).
  - `apply_move(move)`: aplica a jogada do jogador atual, atualiza `last_move` e troca de jogador; em Pente chama `_apply_pente_captures`.
  - `_apply_pente_captures(player_id, move)`: implementa a regra de captura de pares em Pente (padrão P–O–O–P nas 8 direções).
  - `winner()`: devolve `1` ou `2` em caso de vitória, `0` em empate, ou `None` se o jogo continua (usa `_five_in_row` e o contador de capturas).
  - `_five_in_row(player_id)`: deteta sequências de cinco pedras em linhas, colunas e diagonais.
  - `to_ascii() / __str__()`: representação textual do tabuleiro para debugging.

---

### `board_encoding.py` — codificação para redes

- `board_to_tensor(board, my_id, board_size=15)`: converte o tabuleiro num tensor PyTorch `(1, 3, board_size, board_size)` com:
  - canal 0: pedras do jogador atual  
  - canal 1: pedras do adversário  
  - canal 2: plano constante (+1 / −1) a indicar o lado do jogador  

É a função de entrada comum para **CNN**, **ResNet** e os jogadores AlphaZero.

---

### `net.py` — rede CNN tipo AlphaZero

- **`class GomokuNet(nn.Module)`**
  - `__init__(board_size=15, in_channels=3)`: define um “tronco” convolucional com 3 blocos conv+BN+ReLU.
  - `forward(x)`: devolve  
    - `policy_logits` `(batch, 225)`: logits sobre todas as casas do tabuleiro  
    - `value` `(batch, 1)`: avaliação do estado em `[-1, 1]`  
    através de uma cabeça de política e uma cabeça de valor separadas.

---

### `resnet.py` — rede ResNet compacta

- **`class ResidualBlock(nn.Module)`**
  - `forward(x)`: bloco residual padrão (duas conv3×3 + BN + atalho `x + F(x)`).

- **`class ResNetGomoku(nn.Module)`**
  - `__init__(board_size=15, in_channels=3, channels=64, num_blocks=6)`: ResNet pequena com vários `ResidualBlock`.
  - `forward(x)`: tal como `GomokuNet`, devolve `policy_logits` e `value`, mas com um “tronco” residual mais profundo.

---

### `cnn_player.py` — jogador puro CNN

- **`class Player`**
  - `__init__(rules, board_size, model_path=None, device=None, temperature=0.0)`  
    Carrega um modelo `GomokuNet` (`{rules}_cnn.pt` por defeito), define dispositivo e temperatura.
  - `play(board, turn_number, last_opponent_move)`  
    1. Determina o `my_id` a partir do turno  
    2. Codifica o tabuleiro com `board_to_tensor`  
    3. Obtém `policy_logits` da CNN  
    4. Mascara casas ocupadas e calcula probabilidades  
    5. Amostra uma jogada válida de acordo com essas probabilidades.

---

### `resnet_player.py` — jogador puro ResNet

- **`class Player`**
  - Estrutura idêntica ao `cnn_player`, mas usando `ResNetGomoku` e pesos `{rules}_resnet.pt`.
  - `play(...)`: mesma lógica de inferência e amostragem da jogada via ResNet.

---

### `mcts_player.py` — MCTS clássico com rollouts

**Funções de baixo nível (Numba)**

- `_five_in_row(board, player)`, `_board_full(board)`: verificações rápidas de fim de jogo.
- `_apply_move_inplace(board, captures, player, r, c, game_mode)`: aplica uma jogada ao `numpy` array e atualiza capturas em Pente.
- `_evaluate_winner(board, captures, game_mode)`: devolve `1`, `2`, `0` (empate) ou `-1` (jogo continua).  
- `_simulate_rollout(...)`: simula um jogo aleatório até fim ou `rollout_limit` e devolve o vencedor.

**Heurísticas de alto nível**

- `_creates_double_threat(board, player, r, c)`: verifica se a jogada cria duas casas de vitória futuras (double-threat).
- `_forced_move_mcts(board, captures, player, game_mode)`: aplica 6 heurísticas (vitória imediata, bloqueio, double-threat, capturas em Pente) para encontrar jogadas “forçadas” antes de correr MCTS.

**MCTS próprio**

- **`class MCTSNode`**
  - guarda `board`, `captures`, jogador, `children`, `visits`, `wins`, `untried_moves`.
  - `ucb1(parent_visits, c)`: valor UCB1 usado na seleção.

- **`class Player`**
  - `__init__(rules, board_size, iterations=None, rollout_limit=None, exploration=1.41, seed=None, time_limit=4.7)`  
    Configura parâmetros do MCTS, limite de tempo e tipo de jogo.
  - `play(board, turn_number, last_opponent_move)`  
    1. Reconstrói o estado interno com `_state_from_view`  
    2. Aplica `_forced_move_mcts`; se houver jogada forçada, usa essa  
    3. Caso contrário, corre o ciclo MCTS (seleção, expansão, simulação via `_simulate_rollout`, backprop) até esgotar `iterations` ou `time_limit`  
    4. Escolhe a melhor criança por taxa de vitórias.
  - `_expand(node, rng)`, `_rollout(node, player_to_move)`, `_backpropagate(node, winner)`, `_best_child(node)`: helpers para o ciclo MCTS.
  - `_state_from_view(board_view, turn_number, my_id)`: estima o vetor de capturas a partir do nº de pedras.
  - `_legal_moves_from_array(arr)`: devolve todas as casas vazias de um tabuleiro `numpy`.

---

### `mcts_alphazero.py` — MCTS tipo AlphaZero (CNN)

- Versão de MCTS que substitui os rollouts por **avaliação de rede neural** e guarda as visitas para políticas AlphaZero.

**Principais elementos**

- `_creates_double_threat(...)` e `_forced_move_az(...)`: heurísticas de jogada forçada antes de chamar o MCTS.
- **`class AZNode`**: nó de árvore com `board`, `captures`, `to_move`, `prior`, `visits`, `value_sum`.
- **`class Player`**
  - `__init__(..., net=None, model_path=None, iterations=None, time_limit=4.7, c_puct=1.5, temperature=1.0, ...)`  
    Usa rede externa (para treino) ou carrega `{rules}_cnn.pt`.
  - `search_with_policy(board, turn_number, last_opponent_move) -> (move, pi)`  
    Corre o MCTS AlphaZero com limite de tempo/iterações, devolvendo a jogada e o vetor de política baseado nas visitas.
  - `play(...)`: wrapper que devolve só a jogada.
  - `_run_simulation`, `_select_child_puct`, `_evaluate_and_expand`, `_backpropagate`, `_select_move_from_root`: implementação do ciclo MCTS com PUCT e avaliação via `GomokuNet`.
  - `_state_from_view`, `_legal_moves_from_array`: helpers de estado e jogadas legais.

---

### `mcts_alphazero_resnet.py` — MCTS AlphaZero (ResNet)

- Estrutura idêntica a `mcts_alphazero.py`, mas usando `ResNetGomoku` e ficheiros `{rules}_resnet.pt`.  
- A classe `Player` aqui é o motor principal usado em self-play e treino em `train_alphazero.py`.

---

### `play.py` — motor de jogos interativos

- `render_board(board: Board)`: desenha o tabuleiro com índices de linha/coluna.
- **`class HumanPlayer`**
  - `choose_move(...)`: lê uma jogada do teclado, verificando limites e casas ocupadas.
- `load_player(path_or_mod, rules, size, player_id)`: carrega dinamicamente um módulo com classe `Player` ou devolve `HumanPlayer`.
- `timed_play(...)`: chama `player.play(...)` num processo separado com `TIME_LIMIT_SECONDS`, devolvendo erro/timeout se for ultrapassado.
- `run_match(p1_path, p2_path, rules, size, verbose=True)`  
  Corre um jogo completo entre dois jogadores (humanos ou agentes), aplicando `board.apply_move`, respeitando o limite de tempo e verificando o vencedor a cada turno.
- `main(...)`: interface de linha de comandos (`--game`, `--size`, nomes dos jogadores).

---

### `play_vs_mcts.py` — jogos e testes automáticos

- `print_board(board)`: versão simples de impressão do tabuleiro.
- `_play_single_game(rules, p1, p2, game_index, verbose)`  
  Corre um jogo entre dois agentes (`MCTS`, `AlphaZero`, `CNN`, `ResNet`), mantendo um vetor de capturas em Pente e usando `_evaluate_winner`.
- `_run_test_suite(num_games=100, csv_path="test_results.csv")`  
  Faz um round-robin entre vários tipos de agentes (`az_cnn`, `az_resnet`, `mcts`, `cnn`, `resnet`) em Gomoku e Pente, grava resultados num CSV.
- `main(rules="gomoku", num_games=1, testing=False)`  
  Configuração de conveniência para correr séries de jogos (por exemplo, AlphaZero vs MCTS).

---

### `train_alphazero.py` — self-play e treino (ResNet/CNN)

**Self-play**

- `play_one_game(num_iterations=1000, rules="gomoku", time_limit=None, start_player=1)`  
  Joga um jogo AlphaZero-ResNet vs AlphaZero-ResNet, devolvendo:
  - `history`: lista de `(board_perspetiva_jogador, player_id, move)`
  - `winner`: 0, 1 ou 2  
  Usa `_apply_move_inplace` e `_evaluate_winner` para respeitar regras de Pente.

- `generate_dataset(num_games=10, rules="gomoku", mcts_iterations=1000, mcts_time_limit=None)`  
  Gera o dataset `(X, Pi, Z)`:
  - `X`: tensores de estado  
  - `Pi`: política alvo (one-hot da jogada escolhida)  
  - `Z`: resultado do jogo na perspetiva do jogador.

**Treino de modelos**

- `train_single_model(net, X, Pi, Z, save_path, device, ...)`  
  Treina uma rede (CNN ou ResNet) em cima do dataset:
  - loss total = MSE do valor + cross-entropy da política  
  - validação opcional e early-stopping  
  - guarda pesos no ficheiro indicado.

- `train(..., rules="gomoku", train_resnet=True, train_cnn=False)`  
  Combina geração de dados + treino de uma ResNet e/ou CNN, produzindo `{rules}_resnet.pt` e `{rules}_cnn.pt`.

- `train_loop(...)`  
  Corre vários ciclos de:
  1. gerar jogos de self-play  
  2. treinar a ResNet com os dados novos  
  3. alternar entre `gomoku` e `pente` se desejado.

- `train_forever(...)`  
  Versão em loop infinito, útil para deixar o agente a melhorar continuamente até interrupção manual.

---

## 3. Conclusões

Neste projeto implementámos um agente AlphaZero funcional para **Gomoku** e **Pente**, com suporte para diferentes tamanhos de tabuleiro e duas arquiteturas de rede (CNN e ResNet). A combinação de:

- representação explícita das regras de jogo (`board.py`)  
- codificação adequada do estado (`board_encoding.py`)  
- redes neuronais para política e valor (`net.py`, `resnet.py`)  
- pesquisa MCTS guiada pela rede (`mcts_alphazero*.py`)  

permite obter agentes que aprendem a jogar a partir de self-play, sem conhecimento prévio especializado.

O trabalho permitiu consolidar conceitos de:
- representação de estados e regras em jogos de tabuleiro  
- integração entre redes neuronais e algoritmos de pesquisa  
- treino por self-play e avaliação empírica de agentes de jogo

Como trabalho futuro, seria interessante:
- prolongar o treino com mais jogos de self-play e redes maiores  
- explorar data augmentation com simetrias do tabuleiro  
- testar variantes do algoritmo (por exemplo, MuZero) ou combinações híbridas com heurísticas específicas para Gomoku e Pente.
